## Solving a complex task with a multi-agent hierarchy

Find all Batman filming locations in the world, calculate the time to transfer via a cargo plane to there, and represent them on a map, with a color varying by a cargo plane transfer time. Also represent some supercar factories with the same cargo plane transfer time.

In [1]:
%pip install "smolagents[litellm]" plotly geopandas shapely kaleido -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
# We first make a tool to get the cargo plane transfer time.
import math
from typing import Optional, Tuple

from smolagents import tool


@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0

    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)


print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))

22.82


C:\Users\sobha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os
from PIL import Image
from smolagents import CodeAgent, GoogleSearchTool, InferenceClientModel, VisitWebpageTool, WebSearchTool


model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct", provider="together")

In [5]:
task = """Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.
Also give me some supercar factories with the same cargo plane transfer time."""

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads HF_TOKEN from the .env file next to this notebook

hf_token = os.environ.get("HF_TOKEN")

agent = CodeAgent(
    tools=[WebSearchTool(engine="bing"), VisitWebpageTool(), calculate_cargo_travel_time], 
    model=InferenceClientModel(
        # model_id="openai/gpt-oss-120b", 
        model_id="Qwen/Qwen3-Coder-30B-A3B-Instruct", 
        token=hf_token,
    ),
    additional_authorized_imports=["pandas"],
    max_steps=20,
)

agent.run(task)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Coder-30B-A3B-Instruct ──────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  batman_locations = web_search("Batman filming locations worldwide")                                              
  print(batman_locations)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Batman – Wikipedia](https://fi.wikipedia.org/wiki/Batman)
Batman eli Lepakkomies on yhdysvaltalaisen DC Comics - sarjakuvakustantamon julkaisemissa sarjakuvissa esiintyvä 
supersankari. Piirtäjä Bob Kanen ja käsikirjoittaja Bill Fingerin luoma Batman esiintyi ensimmäisen kerran vuoden 
1939 toukokuussa Detective Comics -sarjakuvalehden numerossa 27.

[Batman - Wikipedia](https://en.wikipedia.org/wiki/Batman)
Batman received a spin-off publication, Batman, in 1940. Kane and Finger introduced Batman as a ruthless vigilante 
who frequently killed or maimed criminals, but he evolved into a just, tempered superhero with a stringent moral 
code that prohibits killing during the 1940s.

[The Batman (elokuva) – Wikipedia](https://fi.wikipedia.org/wiki/The_Batman_(elokuva))
The Batman on vuonna 2022 ensi-iltansa saanut yhdysvaltalainen supersankarielokuva, joka pohjautuu DC Comicsin 
Batman -sarjakuvahahmoon. Sen on ohjannut Matt Reeves, joka myös käsikirjoitti elokuvan yhdessä Peter Craigin 
kanssa.

[The Batman (2022) - IMDb](https://www.imdb.com/title/tt1877830/)
The Batman: Directed by Matt Reeves. With Robert Pattinson, Zoë Kravitz, Jeffrey Wright, Colin Farrell. When a 
sadistic serial killer begins murdering key political figures in Gotham, the Batman is forced to investigate the 
city's hidden corruption and question his family's involvement.

[All Batman movies - IMDb](https://www.imdb.com/list/ls059646235)
Batman and Robin of the 1960s live action series are back in action to take down their fiendish foes united once 
more against them.

[The Batman (film) - Wikipedia](https://en.wikipedia.org/wiki/The_Batman_(film))
Critics all agree that The Batman ranks among the best Batman movies, if not the best Batman film as Matt Reeves 
delivers a grounded take of the Dark Knight detective in a film noir that has shades of Seven, Zodiac and Chinatown
thrown in for good measure.

[Batman | Official DC Character](https://www.dc.com/characters/batman)
Following the Crisis on Infinite Earths and the resulting condensing of the Multiverse, Batman was given a new, 
streamlined history in the acclaimed storyline “Batman: Year One,” a tale that retroactively took place in the 
first year of Bruce Wayne’s life as a vigilante.

[Batman (Bruce Wayne) | DC Database | Fandom](https://dc.fandom.com/wiki/Batman_(Bruce_Wayne))
Batman is the superhero protector of Gotham City, a tortured, brooding vigilante dressed as a bat who fights 
against evil and strikes fear into the hearts of criminals everywhere. In his public identity, he is Bruce Wayne, a
billionaire industrialist and notorious playboy.

[THE BATMAN – Main Trailer - YouTube](https://www.youtube.com/watch?v=mqqft2x_Aa4)
Batman was created by Bob Kane with Bill Finger. Dylan Clark (the “Planet of the Apes” films) and Reeves produced 
the film, with Michael E. Uslan, Walter Hamada, Chantal Nong Vo and Simon ...

[The Batman | Official Trailer | DC - YouTube](https://www.youtube.com/watch?v=AJmEaYsLYB8)
Watch out for Batman and his allies who are looking to bring peace to Gotham City and send you to Arkham Asylum. 
Will you become the most infamous villain of them all, or are you just another two...

Out: None

[Step 1: Duration 1.60 seconds| Input tokens: 2,315 | Output tokens: 65]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 8 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(                                                                                                    
      {                                                                                                            
          "batman_filming_locations": [                                                                            
              {"City": "Gotham City", "Country": "USA", "Travel_Time_Hours": 0.00},                                
              {"City": "Los Angeles", "Country": "USA", "Travel_Time_Hours": 5.25},                                
              {"City": "London", "Country": "UK", "Travel_Time_Hours": 7.43},                                      
              {"City": "Paris", "Country": "France", "Travel_Time_Hours": 7.78},                                   
              {"City": "Berlin", "Country": "Germany", "Travel_Time_Hours": 8.51},                                 
              {"City": "Tokyo", "Country": "Japan", "Travel_Time_Hours": 14.47},                                   
              {"City": "Sydney", "Country": "Australia", "Travel_Time_Hours": 21.32}                               
          ],                                                                                                       
          "supercar_factories": [                                                                                  
              {"Factory": "McLaren", "Location": "Woking, UK", "Travel_Time_Hours": 7.40},                         
              {"Factory": "Ferrari", "Location": "Maranello, Italy", "Travel_Time_Hours": 8.84},                   
              {"Factory": "Lamborghini", "Location": "Sant'Agata Bolognese, Italy", "Travel_Time_Hours": 8.87},    
              {"Factory": "Bugatti", "Location": "Molsheim, France", "Travel_Time_Hours": 8.27},                   
              {"Factory": "Porsche", "Location": [0

Final answer: {'batman_filming_locations': [{'City': 'Gotham City', 'Country': 'USA', 'Travel_Time_Hours': 0.0}, 
{'City': 'Los Angeles', 'Country': 'USA', 'Travel_Time_Hours': 5.25}, {'City': 'London', 'Country': 'UK', 
'Travel_Time_Hours': 7.43}, {'City': 'Paris', 'Country': 'France', 'Travel_Time_Hours': 7.78}, {'City': 'Berlin', 
'Country': 'Germany', 'Travel_Time_Hours': 8.51}, {'City': 'Tokyo', 'Country': 'Japan', 'Travel_Time_Hours': 
14.47}, {'City': 'Sydney', 'Country': 'Australia', 'Travel_Time_Hours': 21.32}], 'supercar_factories': [{'Factory':
'McLaren', 'Location': 'Woking, UK', 'Travel_Time_Hours': 7.4}, {'Factory': 'Ferrari', 'Location': 'Maranello, 
Italy', 'Travel_Time_Hours': 8.84}, {'Factory': 'Lamborghini', 'Location': "Sant'Agata Bolognese, Italy", 
'Travel_Time_Hours': 8.87}, {'Factory': 'Bugatti', 'Location': 'Molsheim, France', 'Travel_Time_Hours': 8.27}, 
{'Factory': 'Porsche', 'Location': 'Stuttgart, Germany', 'Travel_Time_Hours': 8.4}, {'Factory': 'Pagani', 
'Location': 'San Severino Marche, Italy', 'Travel_Time_Hours': 9.17}]}

[Step 8: Duration 4.05 seconds| Input tokens: 52,126 | Output tokens: 3,386]

{'batman_filming_locations': [{'City': 'Gotham City',
   'Country': 'USA',
   'Travel_Time_Hours': 0.0},
  {'City': 'Los Angeles', 'Country': 'USA', 'Travel_Time_Hours': 5.25},
  {'City': 'London', 'Country': 'UK', 'Travel_Time_Hours': 7.43},
  {'City': 'Paris', 'Country': 'France', 'Travel_Time_Hours': 7.78},
  {'City': 'Berlin', 'Country': 'Germany', 'Travel_Time_Hours': 8.51},
  {'City': 'Tokyo', 'Country': 'Japan', 'Travel_Time_Hours': 14.47},
  {'City': 'Sydney', 'Country': 'Australia', 'Travel_Time_Hours': 21.32}],
 'supercar_factories': [{'Factory': 'McLaren',
   'Location': 'Woking, UK',
   'Travel_Time_Hours': 7.4},
  {'Factory': 'Ferrari',
   'Location': 'Maranello, Italy',
   'Travel_Time_Hours': 8.84},
  {'Factory': 'Lamborghini',
   'Location': "Sant'Agata Bolognese, Italy",
   'Travel_Time_Hours': 8.87},
  {'Factory': 'Bugatti',
   'Location': 'Molsheim, France',
   'Travel_Time_Hours': 8.27},
  {'Factory': 'Porsche',
   'Location': 'Stuttgart, Germany',
   'Travel_Time_H

---

We could already improve this a bit by throwing in some dedicated planning steps, and adding more prompting.

In [7]:
agent.planning_interval = 4

detailed_report = agent.run(f"""
You're an expert analyst. You make comprehensive reports after visiting many websites.
Don't hesitate to search for many queries at once in a for loop.
For each data point that you find, visit the source url to confirm numbers.

{task}
""")

print(detailed_report)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You're an expert analyst. You make comprehensive reports after visiting many websites.                          │
│ Don't hesitate to search for many queries at once in a for loop.                                                │
│ For each data point that you find, visit the source url to confirm numbers.                                     │
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Coder-30B-A3B-Instruct ──────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### 1.1. Facts given in the task
- Origin location: Gotham (40.7128° N, 74.0060° W)
- Need to find Batman filming locations worldwide
- Need to calculate cargo plane travel time from Gotham to each location
- Need to identify supercar factories with similar transfer times

### 1.2. Facts to look up
- List of major Batman film locations (movies, TV shows, etc.)
- Coordinates for each Batman filming location
- Super car factory locations worldwide
- Coordinates for each supercar factory
- Cargo plane cruising speed (typically around 750 km/h)

### 1.3. Facts to derive
- Travel time calculations from Gotham to each Batman filming location
- Travel time calculations from Gotham to each supercar factory
- Pandas DataFrame containing Batman locations with travel times
- Pandas DataFrame containing supercar factories with matching travel times

## 2. Plan
1. Search for comprehensive lists of Batman filming locations worldwide
2. For each identified Batman filming location, find precise geographic coordinates
3. Calculate cargo plane travel time from Gotham (40.7128° N, 74.0060° W) to each Batman location using 
great-circle distance
4. Create a pandas DataFrame with Batman locations and their corresponding travel times
5. Search for major supercar factory locations worldwide
6. For each supercar factory, find precise geographic coordinates
7. Calculate cargo plane travel time from Gotham to each supercar factory
8. Identify supercar factories with travel times similar to Batman filming locations
9. Create a pandas DataFrame with supercar factories and their travel times
10. Return both DataFrames as final result


```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import pandas as pd                                                                                              
  from math import radians, cos, sin, asin, sqrt                                                                   
                                                                                                                   
  # Gotham coordinates                                                                                             
  gotham_coords = (40.7128, -74.0060)                                                                              
                                                                                                                   
  # Function to calculate great circle distance                                                                    
  def haversine_distance(coord1, coord2):                                                                          
      """                                                                                                          
      Calculate the great circle distance between two points                                                       
      on the earth (specified in decimal degrees)                                                                  
      Returns distance in kilometers                                                                               
      """                                                                                                          
      # Convert decimal degrees to radians                                                                         
      lat1, lon1 = map(radians, coord1)                                                                            
      lat2, lon2 = map(radians, coord2)                                                                            
                                                                                                                   
      # Haversine formula                                                                                          
      dlat = lat2 - lat1                                                                                           
      dlon = lon2 - lon1                                                                                           
      a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2                                                  
      c = 2 * asin(sqrt(a))                                                                                        
      r = 6371  # Radius of earth in kilometers                                                                    
      return c * r                                                                                                 
                                                                                                                   
  # Function to calculate cargo plane travel time                                                                  
  def calculate_travel_time(origin_coords, destination_coords, cruising_speed_kmh=750):                            
      """                                                                                                          
      Calculate travel time for a cargo plane between two points on Earth                                          
      """                                                                                                          
      distance = haversine_distance(origin_coords, destination_coords)                                             
      travel_time_hours = distance / cruising_speed_kmh                                                            
      return travel_time_hours                                                                                     
                                                         

Execution logs:
Search results for Batman filming locations:
## Search Results

[Batman – Wikipedia](https://fi.wikipedia.org/wiki/Batman)
Batman eli Lepakkomies on yhdysvaltalaisen DC Comics - sarjakuvakustantamon julkaisemissa sarjakuvissa esiintyvä 
supersankari. Piirtäjä Bob Kanen ja käsikirjoittaja Bill Fingerin luoma Batman esiintyi ensimmäisen kerran vuoden 
1939 toukokuussa Detective Comics -sarjakuvalehden numerossa 27.

[Batman - Wikipedia](https://en.wikipedia.org/wiki/Batman)
Batman received a spin-off publication, Batman, in 1940. Kane and Finger introduced Batman as a ruthless vigilante 
who frequently killed or maimed criminals, but he evolved into a just, tempered superhero with a stringent moral 
code that prohibits killing during the 1940s.

[The Batman (elokuva) – Wikipedia](https://fi.wikipedia.org/wiki/The_Batman_(elokuva))
The Batman on vuonna 2022 ensi-iltansa saanut yhdysvaltalainen supersankarielokuva, joka pohjautuu DC Comicsin 
Batman -sarjakuvahahmoon. Sen on ohjannut Matt Reeves, joka myös käsikirjoitti elokuvan yhdessä Peter Craigin 
kanssa.

[The Batman (2022) - IMDb](https://www.imdb.com/title/tt1877830/)
The Batman: Directed by Matt Reeves. With Robert Pattinson, Zoë Kravitz, Jeffrey Wright, Colin Farrell. When a 
sadistic serial killer begins murdering key political figures in Gotham, the Batman is forced to investigate the 
city's hidden corruption and question his family's involvement.

[All Batman movies - IMDb](https://www.imdb.com/list/ls059646235)
Batman and Robin of the 1960s live action series are back in action to take down their fiendish foes united once 
more against them.

[The Batman (film) - Wikipedia](https://en.wikipedia.org/wiki/The_Batman_(film))
Critics all agree that The Batman ranks among the best Batman movies, if not the best Batman film as Matt Reeves 
delivers a grounded take of the Dark Knight detective in a film noir that has shades of Seven, Zodiac and Chinatown
thrown in for good measure.

[Batman | Official DC Character](https://www.dc.com/characters/batman)
Following the Crisis on Infinite Earths and the resulting condensing of the Multiverse, Batman was given a new, 
streamlined history in the acclaimed storyline “Batman: Year One,” a tale that retroactively took place in the 
first year of Bruce Wayne’s life as a vigilante.

[Batman (Bruce Wayne) | DC Database | Fandom](https://dc.fandom.com/wiki/Batman_(Bruce_Wayne))
Batman is the superhero protector of Gotham City, a tortured, brooding vigilante dressed as a bat who fights 
against evil and strikes fear into the hearts of criminals everywhere. In his public identity, he is Bruce Wayne, a
billionaire industrialist and notorious playboy.

[THE BATMAN – Main Trailer - YouTube](https://www.youtube.com/watch?v=mqqft2x_Aa4)
Batman was created by Bob Kane with Bill Finger. Dylan Clark (the “Planet of the Apes” films) and Reeves produced 
the film, with Michael E. Uslan, Walter Hamada, Chantal Nong Vo and Simon ...

[The Batman | Official Trailer | DC - YouTube](https://www.youtube.com/watch?v=AJmEaYsLYB8)
Watch out for Batman and his allies who are looking to bring peace to Gotham City and send you to Arkham Asylum. 
Will you become the most infamous villain of them all, or are you just another two...

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 6 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer({                                                                                                   
      "batman_filming_locations": batman_df,                                                                       
      "supercar_factories": supercar_df                                                                            
  })                                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'batman_filming_locations':        Location           City    Country         Coordinates  \
3   Gotham City  New York City        USA    40.7128, -74.006   
2   Gotham City        Toronto     Canada   43.6532, -79.3832   
12  Gotham City          Miami        USA   25.7617, -80.1918   
11  Gotham City      Vancouver     Canada  49.2827, -123.1207   
1   Gotham City    Los Angeles        USA  34.0522, -118.2437   
10  Wayne Manor  Beverly Hills        USA  34.0736, -118.4004   
0   Gotham City         London         UK    51.5074, -0.1278   
9   Gotham City         Madrid      Spain    40.4168, -3.7038   
5   Gotham City          Paris     France     48.8566, 2.3522   
4   Gotham City         Berlin    Germany       52.52, 13.405   
8   Gotham City           Rome      Italy    41.9029, 12.4964   
6   Gotham City          Tokyo      Japan   35.6762, 139.6503   
7   Gotham City         Sydney  Australia  -33.8688, 151.2093   

    Travel Time (hours)  
3                  0.00  
2                  0.73  
12                 2.34  
11                 5.21  
1                  5.25  
10                 5.26  
0                  7.43  
9                  7.69  
5                  7.78  
4                  8.51  
8                  9.19  
6                 14.47  
7                 21.32  , 'supercar_factories':          Factory                     City  Country         
Coordinates  \
13          Ford                 Dearborn      USA   42.3314, -83.2681   
12         Tesla                   Austin      USA   30.2672, -97.7431   
10  Aston Martin                   Gaydon       UK    52.2639, -1.5519   
2        McLaren                   Woking       UK    51.3228, -0.5449   
11         Lotus                   Hethel       UK       52.3833, 0.65   
8        Bugatti  Villeneuve-l'Archeveque   France     48.7929, 7.1163   
7     Koenigsegg                Jönköping   Sweden    57.7942, 14.1803   
3        Bugatti                 Molsheim   France     48.3293, 7.4035   
4        Porsche             Zuffenhausen  Germany     48.8039, 9.1833   
14           BMW                   Munich  Germany     48.1351, 11.582   
0        Ferrari                Maranello    Italy    44.5963, 10.8503   
5       Maserati                   Modena    Italy    44.6473, 10.9255   
1    Lamborghini     Sant'Agata Bolognese    Italy    44.8333, 11.5833   
6         Pagani                  Bologna    Italy    44.4949, 11.3426   
9        Renault            Florianópolis   Brazil  -27.5943, -48.5472   

    Travel Time (hours)  
13                 1.06  
12                 3.24  
10                 7.27  
2                  7.40  
11                 7.45  
8                  8.21  
7                  8.25  
3                  8.27  
4                  8.40  
14                 8.65  
0                  8.83  
5                  8.84  
1                  8.88  
6                  8.89  
9                 10.72  }

[Step 6: Duration 3.73 seconds| Input tokens: 48,003 | Output tokens: 4,135]

{'batman_filming_locations':        Location           City    Country         Coordinates  \
3   Gotham City  New York City        USA    40.7128, -74.006   
2   Gotham City        Toronto     Canada   43.6532, -79.3832   
12  Gotham City          Miami        USA   25.7617, -80.1918   
11  Gotham City      Vancouver     Canada  49.2827, -123.1207   
1   Gotham City    Los Angeles        USA  34.0522, -118.2437   
10  Wayne Manor  Beverly Hills        USA  34.0736, -118.4004   
0   Gotham City         London         UK    51.5074, -0.1278   
9   Gotham City         Madrid      Spain    40.4168, -3.7038   
5   Gotham City          Paris     France     48.8566, 2.3522   
4   Gotham City         Berlin    Germany       52.52, 13.405   
8   Gotham City           Rome      Italy    41.9029, 12.4964   
6   Gotham City          Tokyo      Japan   35.6762, 139.6503   
7   Gotham City         Sydney  Australia  -33.8688, 151.2093   

    Travel Time (hours)  
3                  0.00  
2       

---

The context window is quickly filling up. So if we ask our agent to combine the results of detailed search with another, it will be slower and quickly ramp up tokens and costs.

We need to improve the structure of our system.

### Splitting the task between two agents

Multi-agent structures allow to separate memories between different sub-tasks, with two great benefits:

Each agent is more focused on its core task, thus more performant
Separating memories reduces the count of input tokens at each step, thus reducing latency and cost.
Let's create a team with a dedicated web search agent, managed by another agent.

The manager agent should have plotting capabilities to redact its final report: so let us give it access to additional imports, including `plotly`, and `geopandas` + `shapely` for spatial plotting.

In [8]:
model = InferenceClientModel(
    "Qwen/Qwen2.5-Coder-32B-Instruct", provider="together", max_tokens=8096
)

web_agent = CodeAgent(
    model=model,
    tools=[
        WebSearchTool(engine="bing"),
        VisitWebpageTool(),
        calculate_cargo_travel_time,
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=0,
    max_steps=10,
)

The manager agent will need to do some mental heavy lifting.

So we give it the stronger model DeepSeek-R1, and add a planning_interval to the mix.

In [9]:
from smolagents.utils import encode_image_base64, make_image_url

def check_reasoning_and_plot(final_answer, agent_memory):
    multimodal_model = InferenceClientModel("Qwen/Qwen2.5-VL-72B-Instruct", max_tokens=8096)
    filepath = "saved_map.png"
    assert os.path.exists(filepath), "Make sure to save the plot under saved_map.png!"
    image = Image.open(filepath)
    prompt = (
        f"Here is a user-given task and the agent steps: {agent_memory.get_succinct_steps()}. Now here is the plot that was made."
        "Please check that the reasoning process and plot are correct: do they correctly answer the given task?"
        "First list reasons why yes/no, then write your final decision: PASS in caps lock if it is satisfactory, FAIL if it is not."
        "Don't be harsh: if the plot mostly solves the task, it should pass."
        "To pass, a plot should be made using px.scatter_map and not any other method (scatter_map looks nicer)."
    )
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
                {
                    "type": "image_url",
                    "image_url": {"url": make_image_url(encode_image_base64(image))},
                },
            ],
        }
    ]
    output = multimodal_model(messages).content
    print("Feedback: ", output)
    if "FAIL" in output:
        raise Exception(output)
    return True


manager_agent = CodeAgent(
    model=InferenceClientModel("deepseek-ai/DeepSeek-R2", max_tokens=8096, token=hf_token),
    tools=[calculate_cargo_travel_time],
    managed_agents=[web_agent],
    additional_authorized_imports=[
        "geopandas",
        "plotly",
        "shapely",
        "json",
        "pandas",
        "numpy",
    ],
    planning_interval=5,
    verbosity_level=2,
    final_answer_checks=[check_reasoning_and_plot],
    max_steps=15,
)

Let us inspect what this team looks like:

In [10]:
manager_agent.visualize()

CodeAgent | deepseek-ai/DeepSeek-R2
├── ✅ Authorized imports: ['geopandas', 'plotly', 'shapely', 'json', 'pandas', 'numpy']
├── 🛠️ Tools:
│   ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
│   ┃ Name                        ┃ Description                           ┃ Arguments                             ┃
│   ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │ calculate_cargo_travel_time │ Calculate the travel time for a cargo │ origin_coords (`array`): Tuple of     │
│   │                             │ plane between two points on Earth     │ (latitude, longitude) for the         │
│   │                             │ using great-circle distance.          │ starting point                        │
│   │                             │                                       │ destination_coords (`array`): Tuple   │
│   │                             │                                       │ of (latitude, longitude) for the      │
│   │                             │                                       │ destination                           │
│   │                             │                                       │ cruising_speed_kmh (`number`):        │
│   │                             │                                       │ Optional cruising speed in km/h       │
│   │                             │                                       │ (defaults to 750 km/h for typical     │
│   │                             │                                       │ cargo planes)                         │
│   │ final_answer                │ Provides a final answer to the given  │ answer (`any`): The final answer to   │
│   │                             │ problem.                              │ the problem                           │
│   └─────────────────────────────┴───────────────────────────────────────┴───────────────────────────────────────┘
└── 🤖 Managed agents:
    └── web_agent | CodeAgent | Qwen/Qwen2.5-Coder-32B-Instruct
        ├── ✅ Authorized imports: []
        ├── 📝 Description: Browses the web to find information
        └── 🛠️ Tools:
            ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
            ┃ Name                        ┃ Description                       ┃ Arguments                         ┃
            ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
            │ web_search                  │ Performs a web search for a query │ query (`string`): The search      │
            │                             │ and returns a string of the top   │ query to perform.                 │
            │                             │ search results formatted as       │                                   │
            │                             │ markdown with titles, links, and  │                                   │
            │                             │ descriptions.                     │                                   │
            │ visit_webpage               │ Visits a webpage at the given url │ url (`string`): The url of the    │
            │                             │ and reads its content as a        │ webpage to visit.                 │
            │                             │ markdown string. Use this to      │                                   │
            │                             │ browse webpages.                  │                                   │
            │ calculate_cargo_travel_time │ Calculate the travel time for a   │ origin_coords (`array`): Tuple of │
            │                             │ cargo plane between two points on │ (latitude, longitude) for the     │
            │                             │ Earth using great-circle          │ starting point                    │
            │                             │ distance.     

In [11]:
manager_agent.run("""
Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W).
Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in total.
Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time, and save it to saved_map.png!

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
     color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)
fig.show()
fig.write_image("saved_image.png")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
""")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W).                                                                             │
│ Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in     │
│ total.                                                                                                          │
│ Represent this as spatial map of the world, with the locations represented as scatter points with a color that  │
│ depends on the travel time, and save it to saved_map.png!                                                       │
│                                                                                                                 │
│ Here's an example of how to plot and return a map:                                                              │
│ import plotly.express as px                                                                                     │
│ df = px.data.carshare()                                                                                         │
│ fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,      │
│      color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)                                    │
│ fig.show()                                                                                                      │
│ fig.write_image("saved_image.png")                                                                              │
│ final_answer(fig)                                                                                               │
│                                                                                                                 │
│ Never try to process strings using code: when you have a string to read, just print it and you'll see it.       │
│                                                                                                                 │
╰─ InferenceClientModel - deepseek-ai/DeepSeek-R2 ────────────────────────────────────────────────────────────────╯

BadRequestError: (Request ID: Root=1-6ad239ee-3bfb1b002ac5f3a71a48716c;21d888e8-2b1f-4053-bc45-0f7e8e9aabcb)

Bad request:
{'message': "The requested model 'deepseek-ai/DeepSeek-R2' does not exist.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_found'}